### 1-1. 수리 모형 기반 풀이 

In [40]:
from ortools.linear_solver import pywraplp
solver=pywraplp.Solver.CreateSolver('SCIP')
def create_model():
    data={}
    data['distance']=[[0,125,225,155,215],
                      [125,0,85,115,135],
                      [225,85,0,165,190],
                      [155,112,165,0,195],
                      [215,135,190,195,0]]
    data['city']=['Basin','Wald','Bon','Mena','Kiln']
    data['vehicle']=1
    data['depot']=0
    return data
    
def print_solution(data,Data,x,status):
    if status==pywraplp.Solver.OPTIMAL:
        print(f'Total distance = {solver.Objective().Value()}')
        for i in range(Data):
            for j in range(Data):
                if i!=j and x[i,j].solution_value() !=0:
                    print(data['city'][i],'->',data['city'][j])

def main():
    data=create_model()
    x={}
    u={}
    Data=len(data['distance'])
    for i in range(Data):
        u[i]=solver.IntVar(0,solver.infinity(),'u[%i]'%i)
        for j in range(Data):
            if i!=j:
                x[i,j]= solver.BoolVar(f'x[{i},{j}]')
    
    for i in range(Data):
        solver.Add(sum(x[i,j] for j in range(Data) if i!=j)==1)
    for j in range(Data):
        solver.Add(sum(x[i,j] for i in range(Data) if i!=j)==1)
    for i in range(1,Data):
        for j in range(1,Data):
            if i!=j:
                solver.Add(u[i]-u[j]+Data*x[i,j]<=(Data-1))
    solver.Add(u[0]==0)
    obj=[]
    for i in range(Data):
        for j in range(Data):
            if i!=j:
                obj.append(data['distance'][i][j]*x[i,j])
    solver.Minimize(sum(obj))
    status=solver.Solve()
    return print_solution(data,Data,x,status)

if __name__=='__main__':
    main()

Total distance = 750.0
Basin -> Mena
Wald -> Basin
Bon -> Wald
Mena -> Kiln
Kiln -> Bon


### 1-2. Ortools Code 활용 풀이 

In [41]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
def create_model():
    data={}
    M=1000000
    data['distance']=[[0,125,225,155,215],
                      [125,0,85,115,135],
                      [225,85,0,165,190],
                      [155,112,165,0,195],
                      [215,135,190,195,0]]
    data['city']=['Basin','Wald','Bon','Mena','Kiln']
    data['vehicle']=1
    data['depot']=0
    return data

def print_solution(data,manager,routing,solution):
    print(f'Total distance : {solution.ObjectiveValue()}')
    index=routing.Start(0)
    plan_output =f'Routing of Vehicle 0 :\n'
    route_distance=0
    while not routing.IsEnd(index):
        plan_output+='{}({})->'.format(data['city'][manager.IndexToNode(index)],manager.IndexToNode(index))
        previous_index=index
        index=solution.Value(routing.NextVar(index))
        route_distance+=routing.GetArcCostForVehicle(
                previous_index,index,0
            )
    plan_output+='{}({})\n'.format(data['city'][manager.IndexToNode(index)],manager.IndexToNode(index))
    plan_output+=f'Distance of Route: {route_distance}'
    print(plan_output)

def main():
    data=create_model()
    manager=pywrapcp.RoutingIndexManager(
        len(data['distance']),data['vehicle'],data['depot']
    )
    routing=pywrapcp.RoutingModel(manager)

    def distance_callback(from_index,to_index):
        from_node=manager.IndexToNode(from_index)
        to_node=manager.IndexToNode(to_index)
        return data['distance'][from_node][to_node]
    transit_callback_index=routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    search_parameter=pywrapcp.DefaultRoutingSearchParameters()
    search_parameter.first_solution_strategy= routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC


    solution=routing.SolveWithParameters(search_parameter)
    if solution:
        print_solution(data,manager,routing,solution)


if __name__=='__main__':
    main()

Total distance : 750
Routing of Vehicle 0 :
Basin(0)->Mena(3)->Kiln(4)->Bon(2)->Wald(1)->Basin(0)
Distance of Route: 750


### 1-3. Nearest Neighbor 경로 계산   
출발: Basin   
->Wald 선택(거리:125)   
->Bon 선택(거리: 85)   
->Mena 선택(거리: 165)   
->Kiln 선택(거리: 195)   
->Basin 복귀(거리: 215)   

In [42]:
print(f'Total distance: {125+85+165+195+215}')

Total distance: 785


경로 1,2의 최적해인 750보다 더 긴 경로이다 

### 2. Open Tour

In [43]:
DIST = [
       [0, 20, 30, 25, 12, 33, 44, 57, 0],
       [22, 0, 19, 20, 20, 29, 43, 45, 0],
       [28, 19, 0, 17, 38, 48, 55, 60, 0],
       [25, 20, 19, 0, 28, 35, 40, 55, 0],
       [12, 18, 34, 25, 0, 21, 30, 40, 0],
       [35, 25, 45, 30, 20, 0, 25, 39, 0],
       [47, 39, 50, 35, 28, 20, 0, 28, 0],
       [60, 38, 54, 50, 33, 40, 25, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0]
       ]


from ortools.linear_solver import pywraplp

nCity = len(DIST)

solver = pywraplp.Solver.CreateSolver("SAT")

x = {}
for i in range(nCity):
    for j in range(nCity):
        if i != j:
            x[i, j] = solver.IntVar(0, 1, "x"+str(i)+str(j))

u = {}
for i in range(1, nCity):
    u[i] = solver.IntVar(1, nCity-1, "u[%i]" %i)
    
# 제약조건

for j in range(1, nCity):
    con = [x[i, j] for i in range(nCity) if i != j]
    solver.Add(solver.Sum(con) == 1)

for i in range(1, nCity):
    con = [x[i, j] for j in range(nCity) if i != j]
    solver.Add(solver.Sum(con) == 1)
    

for i in range(1, nCity):
    for j in range(1, nCity):
        if i != j:
            solver.Add(u[i] - u[j] + nCity * x[i, j] <= nCity - 1)

solver.Add(u[6] == nCity - 1)

objective_terms = []
for i in range(nCity):
    for j in range(nCity):
        if i != j:
            objective_terms.append(DIST[i][j] * x[i, j])

solver.Minimize(solver.Sum(objective_terms))

        
# Solve
status = solver.Solve()

# Print solution.
if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
    print(f"Total cost = {solver.Objective().Value():.1f}\n", ) 
    for i in range(nCity):
        for j in range(nCity):
            if i != j:
                if x[i, j].solution_value() > 0.5:
                    print(f"X{i} -> X{j}")
    for i in range(1, nCity):
        print(f"{i} 도시 순서: ", u[i].solution_value())

Total cost = 166.0

X0 -> X4
X1 -> X2
X2 -> X3
X3 -> X8
X4 -> X5
X5 -> X1
X6 -> X0
X7 -> X6
X8 -> X7
1 도시 순서:  3.0
2 도시 순서:  4.0
3 도시 순서:  5.0
4 도시 순서:  1.0
5 도시 순서:  2.0
6 도시 순서:  8.0
7 도시 순서:  7.0
8 도시 순서:  6.0


### 3. TSP 서비스센터 

In [44]:
DIST = [
       [0, 20, 15, 19, 24, 14, 21, 11],
       [20, 0, 18, 22, 23, 22, 9, 10],
       [15, 18, 0, 11, 21, 14, 32, 12],
       [19, 22, 11, 0, 20, 27, 18, 15],
       [24, 23, 21, 20, 0, 14, 25, 20],
       [14, 22, 14, 27, 14, 0, 26, 17],
       [21, 9, 32, 18, 25, 26, 0, 20],
       [11, 10, 12, 15, 20, 17, 20, 0]
       ]


from ortools.linear_solver import pywraplp

nCity = len(DIST)

solver = pywraplp.Solver.CreateSolver("SAT")

# 의사결정변수

x = {}
for i in range(nCity):
    for j in range(nCity):
        if i != j:
            x[i, j] = solver.IntVar(0, 1, "x"+str(i)+str(j))

u = {}
for i in range(1, nCity):
    u[i] = solver.IntVar(1, nCity-1, "u[%i]" %i)
    
# 제약조건

for j in range(nCity):
    con = [x[i, j] for i in range(nCity) if i != j]
    solver.Add(solver.Sum(con) == 1)

for i in range(nCity):
    con = [x[i, j] for j in range(nCity) if i != j]
    solver.Add(solver.Sum(con) == 1)


# 방문 순서 제약
for i in range(1, nCity):
    for j in range(1, nCity):
        if i != j:
            solver.Add(u[i] - u[j] + nCity * x[i, j] <= nCity - 1)

# Objective
objective_terms = []
for i in range(nCity):
    for j in range(nCity):
        if i != j:
            objective_terms.append(DIST[i][j] * x[i, j])

solver.Minimize(solver.Sum(objective_terms))

        
# Solve
status = solver.Solve()

# Print solution.
if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
    print(f"Total cost = {solver.Objective().Value():.1f}\n", ) 
    for i in range(nCity):
        for j in range(nCity):
            if i != j:
                if x[i, j].solution_value() > 0.5:
                    print(f"X{i} --> X{j}")
    for i in range(1, nCity):
        print(f"{i} 도시 순서 : ", u[i].solution_value())
else:
    print("No solution found.")

Total cost = 108.0

X0 --> X5
X1 --> X7
X2 --> X3
X3 --> X6
X4 --> X2
X5 --> X4
X6 --> X1
X7 --> X0
1 도시 순서 :  6.0
2 도시 순서 :  3.0
3 도시 순서 :  4.0
4 도시 순서 :  2.0
5 도시 순서 :  1.0
6 도시 순서 :  5.0
7 도시 순서 :  7.0


In [45]:
DIST = [
        [0, 20, 15, 19, 24, 14, 21, 11],
        [20, 0, 18, 22, 23, 22, 9, 10],
        [15, 18, 0, 11, 21, 14, 32, 12],
        [19, 22, 11, 0, 20, 27, 18, 15],
        [24, 23, 21, 20, 0, 14, 25, 20],
        [14, 22, 14, 27, 14, 0, 26, 17],
        [21, 9, 32, 18, 25, 26, 0, 20],
        [11, 10, 12, 15, 20, 17, 20, 0]
        ]


from ortools.linear_solver import pywraplp

nCity = len(DIST)

solver = pywraplp.Solver.CreateSolver("SAT")

# 의사결정변수

x = {}
for i in range(nCity):
    for j in range(nCity):
        if i != j:
            x[i, j] = solver.IntVar(0, 1, "x"+str(i)+str(j))

u = {}
for i in range(1, nCity):
    u[i] = solver.IntVar(1, nCity-1, "u[%i]" %i)
    
# 제약조건


for j in range(nCity):
    con = [x[i, j] for i in range(nCity) if i != j]
    solver.Add(solver.Sum(con) == 1)

for i in range(nCity):
    con = [x[i, j] for j in range(nCity) if i != j]
    solver.Add(solver.Sum(con) == 1)

# 추가 제약조건
solver.Add(u[3] == 1)
solver.Add(u[2] == 2)
solver.Add(u[6] == 3)

# 방문 순서 제약
for i in range(1, nCity):
    for j in range(1, nCity):
        if i != j:
            solver.Add(u[i]-u[j] + 1 - (nCity-1)*(1-x[i, j]) <= 0)


# Objective
objective_terms = []
for i in range(nCity):
    for j in range(nCity):
        if i != j:
            objective_terms.append(DIST[i][j] * x[i, j])

solver.Minimize(solver.Sum(objective_terms))

        
# Solve
status = solver.Solve()

# Print solution.
if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
    print(f"Total cost = {solver.Objective().Value():.1f}\n", ) 
    for i in range(nCity):
        for j in range(nCity):
            if i != j:
                if x[i, j].solution_value() > 0.5:
                    print(f"X{i} --> X{j}")
    for i in range(1, nCity):
        print(f"{i} 도시 순서 : ", u[i].solution_value())
else:
    print("No solution found.")

Total cost = 129.0

X0 --> X3
X1 --> X7
X2 --> X6
X3 --> X2
X4 --> X5
X5 --> X0
X6 --> X1
X7 --> X4
1 도시 순서 :  4.0
2 도시 순서 :  2.0
3 도시 순서 :  1.0
4 도시 순서 :  6.0
5 도시 순서 :  7.0
6 도시 순서 :  3.0
7 도시 순서 :  5.0


### 4.TSP-미팅순서

In [46]:
projects_per_employee = {
    1: [1, 3, 6],
    2: [2, 4, 6],
    3: [1, 5],
    4: [1, 2, 6],
    5: [4, 5, 6],
    6: [5],
    7: [3, 4],
    8: [2, 5, 6],
    9: [1, 2, 3, 4],
    10: [4]
}

nProjects = 6
DIST = [[0]*nProjects for _ in range(nProjects)]

# 각 프로젝트 간의 거리 계산
for i in range(nProjects):
    for j in range(nProjects):
        if i != j:
            employees_i = []
            employees_j = []
            # i번 프로젝트에 참여하는 직원 찾기
            for emp, projects in projects_per_employee.items():
                if i+1 in projects:
                    employees_i.append(emp)
            # j번 프로젝트에 참여하는 직원 찾기
            for emp, projects in projects_per_employee.items():
                if j+1 in projects:
                    employees_j.append(emp)
            # 공통으로 참여하는 직원 수 계산
            common_employees = [emp for emp in employees_i if emp in employees_j]
            similarity = len(common_employees) / max(len(employees_i), len(employees_j))
            DIST[i][j] = 1 - similarity  # 거리는 유사도의 역수로 설정

print("Distance Matrix:")
for row in DIST:
    print([f"{val:.2f}" for val in row])

# (2)


from ortools.linear_solver import pywraplp

nCity = len(DIST)

solver = pywraplp.Solver.CreateSolver("SAT")

# 의사결정변수

x = {}
for i in range(nCity):
    for j in range(nCity):
        if i != j:
            x[i, j] = solver.IntVar(0, 1, "x"+str(i)+str(j))

u = {}
for i in range(1, nCity):
    u[i] = solver.IntVar(1, nCity-1, "u[%i]" %i)
    
# 제약조건
for j in range(1, nCity):
    con = [x[i, j] for i in range(nCity) if i != j]
    solver.Add(solver.Sum(con) == 1)

for i in range(1, nCity):
    con = [x[i, j] for j in range(nCity) if i != j]
    solver.Add(solver.Sum(con) == 1)

# 방문 순서 제약
for i in range(1, nCity):
    for j in range(1, nCity):
        if i != j:
            solver.Add(u[i] - u[j] + nCity * x[i, j] <= nCity - 1)
            
# Objective
objective_terms = []
for i in range(nCity):
    for j in range(nCity):
        if i != j:
            objective_terms.append(DIST[i][j] * x[i, j])

solver.Minimize(solver.Sum(objective_terms))

        
# Solve
status = solver.Solve()

# Print solution.
if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
    print(f"Total cost = {solver.Objective().Value():.1f}\n", ) 
    for i in range(nCity):
        for j in range(nCity):
            if i != j:
                if x[i, j].solution_value() > 0.5:
                    print(f"X{i} --> X{j}")
    for i in range(1, nCity):
        print(f"{i} 도시 순서: ", u[i].solution_value())
else:
    print("No solution found.")

Distance Matrix:
['0.00', '0.50', '0.50', '0.80', '0.75', '0.60']
['0.50', '0.00', '0.75', '0.60', '0.75', '0.40']
['0.50', '0.75', '0.00', '0.60', '1.00', '0.80']
['0.80', '0.60', '0.60', '0.00', '0.80', '0.60']
['0.75', '0.75', '1.00', '0.80', '0.00', '0.60']
['0.60', '0.40', '0.80', '0.60', '0.60', '0.00']
Total cost = 3.4

X0 --> X2
X1 --> X0
X2 --> X3
X3 --> X4
X4 --> X5
X5 --> X1
1 도시 순서:  5.0
2 도시 순서:  1.0
3 도시 순서:  2.0
4 도시 순서:  3.0
5 도시 순서:  4.0


### 5. TSP- 식사배달

In [47]:

DIST = [
       [0, 10, 12, 5, 17, 9, 13, 7],
       [10, 0, 9, 20, 8, 11, 3, 5],
       [12, 9, 0, 14, 4, 10, 1, 16],
       [5, 20, 14, 0, 20, 5, 28, 10],
       [17, 8, 4, 20, 0, 21, 4, 9],
       [9, 11, 10, 5, 21, 0, 2, 3],
       [13, 3, 1, 28, 4, 2, 0, 2],
       [7, 5, 16, 10, 9, 3, 2, 0]
       ]


from ortools.linear_solver import pywraplp

nCity = len(DIST)

solver = pywraplp.Solver.CreateSolver("SAT")

# 의사결정변수

x = {}
for i in range(nCity):
    for j in range(nCity):
        if i != j:
            x[i, j] = solver.IntVar(0, 1, "x"+str(i)+str(j))

u = {}
for i in range(1, nCity):
    u[i] = solver.IntVar(1, nCity-1, "u[%i]" %i)
    
# 제약조건

for j in range(1, nCity):
    con = [x[i, j] for i in range(nCity) if i != j]
    solver.Add(solver.Sum(con) == 1)

for i in range(1, nCity):
    con = [x[i, j] for j in range(nCity) if i != j]
    solver.Add(solver.Sum(con) == 1)

# 방문 순서 제약
for i in range(1, nCity):
    for j in range(1, nCity):
        if i != j:
            solver.Add(u[i] - u[j] + nCity * x[i, j] <= nCity - 1)
            
# Objective
objective_terms = []
for i in range(nCity):
    for j in range(nCity):
        if i != j:
            objective_terms.append(DIST[i][j] * x[i, j])

solver.Minimize(solver.Sum(objective_terms))

        
# Solve
status = solver.Solve()

# Print solution.
if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
    print(f"Total cost = {solver.Objective().Value():.1f}\n", ) 
    for i in range(nCity):
        for j in range(nCity):
            if i != j:
                if x[i, j].solution_value() > 0.5:
                    print(f"X{i} --> X{j}")
    for i in range(1, nCity):
        print(f"{i} 도시 순서: ", u[i].solution_value())
else:
    print("No solution found.")

Total cost = 37.0

X0 --> X3
X1 --> X7
X2 --> X4
X3 --> X5
X4 --> X1
X5 --> X6
X6 --> X2
X7 --> X0
1 도시 순서:  6.0
2 도시 순서:  4.0
3 도시 순서:  1.0
4 도시 순서:  5.0
5 도시 순서:  2.0
6 도시 순서:  3.0
7 도시 순서:  7.0


20분 안에 배송이 불가능하다 